In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import find_peaks, peak_widths, savgol_filter

In [ ]:
def analyze_node_profile(
    csv_file,
    distance_col=0,
    intensity_col=1,
    smooth_window=5,
    polyorder=3,
    prominence=20,
    min_peak_distance_um=1.0,
    plot=True
):
    """
    Robust detection of exactly one peak per paranode.
    """

    df = pd.read_csv(csv_file)

    x = df.iloc[:, distance_col].values
    y_raw = df.iloc[:, intensity_col].values

    dx = x[1] - x[0]

    # Smooth signal
    y = savgol_filter(y_raw, smooth_window, polyorder)

    # Convert min distance from um to points
    min_distance_pts = int(min_peak_distance_um / dx)

    # Detect peaks with minimum spacing
    peaks, properties = find_peaks(
        y,
        prominence=prominence,
        distance=min_distance_pts
    )

    if len(peaks) < 2:
        raise ValueError(f"{csv_file}: less than 2 peaks detected")

    # Choose the two most prominent peaks
    prominences = properties["prominences"]
    best_two_idx = np.argsort(prominences)[-2:]
    selected_peaks = np.sort(peaks[best_two_idx])

    # FWHM
    widths, height, left_ips, right_ips = peak_widths(
        y,
        selected_peaks,
        rel_height=0.5
    )

    left_paranode = widths[0] * dx
    right_paranode = widths[1] * dx

    # Node length = gap between inner half-max points
    node_length = (left_ips[1] - right_ips[0]) * dx

    results = {
        "file": os.path.basename(csv_file),
        "left_paranode_um": left_paranode,
        "right_paranode_um": right_paranode,
        "node_length_um": node_length,
        "left_peak_position": x[selected_peaks[0]],
        "right_peak_position": x[selected_peaks[1]]
    }

    if plot:
        plt.figure(figsize=(8,4))

        plt.plot(x, y_raw, alpha=0.4, label="raw")
        plt.plot(x, y, linewidth=2, label="smoothed")

        plt.plot(x[selected_peaks], y[selected_peaks], "ro")

        for i in range(2):
            plt.hlines(
                y[selected_peaks[i]] / 2,
                x[int(left_ips[i])],
                x[int(right_ips[i])],
                linestyles="dashed"
            )

        plt.axvspan(
            x[int(right_ips[0])],
            x[int(left_ips[1])],
            alpha=0.3,
            label="node"
        )

        plt.xlabel("Distance (µm)")
        plt.ylabel("Intensity")
        plt.legend()
        plt.title(os.path.basename(csv_file))
        plt.show()

    return results

In [ ]:
csv_file = "/content/drive/MyDrive/"

result = analyze_node_profile(csv_file, plot=True)

print(result)